In [2]:
import h5py
import numpy as np
import pandas as pd
import optuna
import torch

from dataclasses import dataclass
from typing import Sequence, Dict, Tuple

from geopy.distance import geodesic
from pyproj import CRS, Transformer
from obspy import Stream, Trace, UTCDateTime

import seisbench.models as sbm
from gamma.utils import association
import json

In [ ]:
# %pip install git+https://github.com/AI4EPS/GaMMA.git

In [3]:
SOURCE_ID_COL = "source_id"
GT_TIME_COL = "source_origin_time"
GT_LAT_COL = "source_latitude"
GT_LON_COL = "source_longitude"
GT_DEPTH_COL = "source_depth_km"

H5_PATH = "data/chunk4.hdf5"
CHANNELS = ("HHZ", "HHN", "HHE")   # matches your ENZ -> ZNE reorder intent
DEFAULT_SAMPLE_RATE = 100.0        # change if your metadata has a different value

In [9]:
def station_id(row) -> str:
    return f"{row.network_code}.{row.receiver_code}."


def build_utm_transformer(meta: pd.DataFrame) -> Tuple[Transformer, Transformer]:
    lat0 = float(meta["receiver_latitude"].median())
    lon0 = float(meta["receiver_longitude"].median())
    zone = int((lon0 + 180) // 6) + 1
    south = " +south" if lat0 < 0 else ""
    utm = CRS.from_proj4(
        f"+proj=utm +zone={zone}{south} +datum=WGS84 +units=m +no_defs"
    )
    fwd = Transformer.from_crs("EPSG:4326", utm, always_xy=True)
    inv = Transformer.from_crs(utm, "EPSG:4326", always_xy=True)
    return fwd, inv


def build_station_df(meta_event: pd.DataFrame, fwd: Transformer) -> pd.DataFrame:
    station_df = (
        meta_event[
            [
                "network_code",
                "receiver_code",
                "receiver_latitude",
                "receiver_longitude",
                "receiver_elevation_m",
            ]
        ]
        .drop_duplicates(subset=["network_code", "receiver_code"])
        .copy()
    )
    station_df["id"] = station_df.apply(station_id, axis=1)
    xy = station_df.apply(
        lambda r: fwd.transform(float(r["receiver_longitude"]), float(r["receiver_latitude"])),
        axis=1,
        result_type="expand",
    )
    station_df["x(km)"] = xy[0] / 1000.0
    station_df["y(km)"] = xy[1] / 1000.0
    station_df["z(km)"] = -station_df["receiver_elevation_m"].astype(float) / 1000.0

    return station_df[["id", "receiver_longitude", "receiver_latitude", "receiver_elevation_m", "x(km)", "y(km)", "z(km)"]]


def load_waveform(trace_name: str) -> np.ndarray:
    with h5py.File(H5_PATH, "r") as h5:
        x = np.asarray(h5["data"][trace_name])
    return x[:, [2, 1, 0]].T  # ENZ -> ZNE, shape: (3, T)


def build_stream(meta_event: pd.DataFrame, sample_rate: float = DEFAULT_SAMPLE_RATE) -> Stream:
    start_time = UTCDateTime(pd.to_datetime(meta_event["trace_start_time"].min()).to_pydatetime())
    st = Stream()

    for row in meta_event.drop_duplicates(subset=["network_code", "receiver_code"]).itertuples():
        x = load_waveform(row.trace_name)
        for comp, data in zip(CHANNELS, x):
            tr = Trace(data=np.asarray(data, dtype=np.float32))
            tr.stats.network = str(row.network_code)
            tr.stats.station = str(row.receiver_code)
            tr.stats.location = ""
            tr.stats.channel = comp
            tr.stats.starttime = start_time
            tr.stats.sampling_rate = sample_rate
            st += tr

    return st


def build_gamma_config(station_df: pd.DataFrame, params) -> dict:
    xmin, xmax = station_df["x(km)"].min() - 20.0, station_df["x(km)"].max() + 20.0
    ymin, ymax = station_df["y(km)"].min() - 20.0, station_df["y(km)"].max() + 20.0

    return {
        "dims": ["x(km)", "y(km)", "z(km)"],
        "use_dbscan": True,
        "use_amplitude": False,
        "x(km)": (xmin, xmax),
        "y(km)": (ymin, ymax),
        "z(km)": (0.0, 150.0),
        "vel": {"p": 7.0, "s": 7.0 / 1.75},
        "method": "BGMM",
        "oversample_factor": 4,
        "bfgs_bounds": (
            (xmin - 1.0, xmax + 1.0),
            (ymin - 1.0, ymax + 1.0),
            (0.0, 151.0),
            (None, None),
        ),
        "dbscan_eps": params.dbscan_eps,
        "dbscan_min_samples": params.dbscan_min_samples,
        "min_picks_per_eq": params.min_picks_per_eq,
        "max_sigma11": params.max_sigma11,
        "max_sigma22": params.max_sigma22,
        "max_sigma12": params.max_sigma12,
    }


# ----------------------------
# params
# ----------------------------

@dataclass(frozen=True)
class SBParams:
    P_threshold: float
    S_threshold: float
    dbscan_eps: int
    dbscan_min_samples: int
    min_picks_per_eq: int
    max_sigma11: float = 2.0
    max_sigma22: float = 1.0
    max_sigma12: float = 1.0


# ----------------------------
# prediction
# ----------------------------

def seisbench_preds(
    meta: pd.DataFrame,
    source_id,
    picker,
    params: SBParams,
    sample_rate: float = DEFAULT_SAMPLE_RATE,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    meta_event = meta[meta.source_id == source_id].drop_duplicates(
        subset=[
            "source_id",
            "network_code",
            "receiver_code",
            "receiver_latitude",
            "receiver_longitude",
            "receiver_elevation_m",
        ]
    ).copy()

    fwd, inv = build_utm_transformer(meta_event)
    station_df = build_station_df(meta_event, fwd)
    stream = build_stream(meta_event, sample_rate=sample_rate)

    classify_output = picker.classify(
        stream,
        batch_size=256,
        P_threshold=params.P_threshold,
        S_threshold=params.S_threshold,
        parallelism=1,
        strict=False,
    )

    picks = classify_output.picks

    pick_df = pd.DataFrame(
        [
            {
                "id": ".".join(p.trace_id.split(".")[:2]) + ".",
                "timestamp": p.peak_time.datetime,
                "prob": p.peak_value,
                "type": p.phase.lower(),
            }
            for p in picks
        ]
    )

    if len(pick_df) == 0:
        return pd.DataFrame(), pick_df

    config = build_gamma_config(station_df, params)
    catalogs, assignments = association(pick_df, station_df, config, method=config["method"])
    catalog = pd.DataFrame(catalogs)

    if len(catalog) == 0:
        return pd.DataFrame(), pick_df

    pred_df = pd.DataFrame(
        {
            "pred_time": pd.to_datetime(catalog["time"], errors="coerce"),
            "pred_lon": catalog.apply(
                lambda r: inv.transform(float(r["x(km)"]) * 1000.0, float(r["y(km)"]) * 1000.0)[0],
                axis=1,
            ),
            "pred_lat": catalog.apply(
                lambda r: inv.transform(float(r["x(km)"]) * 1000.0, float(r["y(km)"]) * 1000.0)[1],
                axis=1,
            ),
            "pred_depth": catalog["z(km)"].astype(float),
        }
    )

    return pred_df.dropna(subset=["pred_time", "pred_lat", "pred_lon"]), pick_df


# ----------------------------
# evaluation
# ----------------------------

def geodesic_km(lat1, lon1, lat2, lon2) -> float:
    return geodesic((float(lat1), float(lon1)), (float(lat2), float(lon2))).km


def build_gt(meta: pd.DataFrame, source_ids: Sequence) -> pd.DataFrame:
    gt = (
        meta.loc[
            meta[SOURCE_ID_COL].isin(source_ids),
            [SOURCE_ID_COL, GT_TIME_COL, GT_LAT_COL, GT_LON_COL, GT_DEPTH_COL],
        ]
        .drop_duplicates(SOURCE_ID_COL)
        .copy()
    )
    gt[GT_TIME_COL] = pd.to_datetime(gt[GT_TIME_COL], errors="coerce")
    gt = gt.dropna(subset=[GT_TIME_COL, GT_LAT_COL, GT_LON_COL])
    return gt.rename(
        columns={
            SOURCE_ID_COL: "source_id",
            GT_TIME_COL: "gt_time",
            GT_LAT_COL: "gt_lat",
            GT_LON_COL: "gt_lon",
            GT_DEPTH_COL: "gt_depth",
        }
    ).reset_index(drop=True)


def score_one_source(gt_row: pd.Series, pred_df: pd.DataFrame) -> Dict[str, float]:
    n_pred = len(pred_df)
    count_penalty = abs(n_pred - 1)

    if n_pred == 0:
        return {
            "count_penalty": count_penalty,
            "best_dt_sec": 1e6,
            "best_dist_km": 1e6,
            "best_depth_km": 1e6,
        }

    gt_time = gt_row["gt_time"]
    gt_lat = gt_row["gt_lat"]
    gt_lon = gt_row["gt_lon"]
    gt_depth = gt_row["gt_depth"]

    best_score = 1e18
    best_dt = 1e6
    best_dist = 1e6
    best_depth = 1e6

    for r in pred_df.itertuples(index=False):
        dt = abs((r.pred_time - gt_time).total_seconds())
        dist = geodesic_km(gt_lat, gt_lon, r.pred_lat, r.pred_lon)
        dep = abs(float(r.pred_depth) - float(gt_depth)) if pd.notna(gt_depth) else 0.0
        score = dt + dist + dep
        if score < best_score:
            best_score = score
            best_dt = dt
            best_dist = dist
            best_depth = dep

    return {
        "count_penalty": count_penalty,
        "best_dt_sec": best_dt,
        "best_dist_km": best_dist,
        "best_depth_km": best_depth,
    }


def evaluate_seisbench(meta, source_ids, picker, params, sample_rate=DEFAULT_SAMPLE_RATE):
    gt = build_gt(meta, source_ids).set_index("source_id")

    count_penalties, dt_list, dist_list, depth_list = [], [], [], []

    for sid in source_ids:
        if sid not in gt.index:
            continue

        pred_df, _ = seisbench_preds(
            meta=meta,
            source_id=sid,
            picker=picker,
            params=params,
            sample_rate=sample_rate,
        )
        m = score_one_source(gt.loc[sid], pred_df)

        count_penalties.append(m["count_penalty"])
        dt_list.append(m["best_dt_sec"])
        dist_list.append(m["best_dist_km"])
        depth_list.append(m["best_depth_km"])

    metrics = {
        "count_penalty": float(np.median(count_penalties)),
        "time_mae_sec": float(np.median(dt_list)),
        "loc_mae_km": float(np.median(dist_list)),
        "depth_mae_km": float(np.median(depth_list)),
    }

    per_source = pd.DataFrame(
        {
            "count_penalty": count_penalties,
            "dt": dt_list,
            "dist": dist_list,
            "depth": depth_list,
        }
    )
    return metrics, per_source


def catalog_objective(metrics):
    return (
        1000.0 * metrics["count_penalty"]
        + min(metrics["time_mae_sec"], 30.0) / 10.0
        + min(metrics["loc_mae_km"], 500.0) / 50.0
        + min(metrics["depth_mae_km"], 100.0) / 20.0
    )


# ----------------------------
# optuna
# ----------------------------

def objective(trial: optuna.Trial, meta, val_ids, picker, sample_rate=DEFAULT_SAMPLE_RATE) -> float:
    params = SBParams(
        P_threshold=trial.suggest_float("P_threshold", 0.03, 0.25),
        S_threshold=trial.suggest_float("S_threshold", 0.03, 0.25),
        dbscan_eps=trial.suggest_int("dbscan_eps", 10, 40),
        dbscan_min_samples=trial.suggest_int("dbscan_min_samples", 2, 6),
        min_picks_per_eq=trial.suggest_int("min_picks_per_eq", 3, 10),
    )

    metrics, _ = evaluate_seisbench(meta, val_ids, picker, params, sample_rate=sample_rate)
    trial.set_user_attr("metrics", metrics)
    return catalog_objective(metrics)


def optimize_seisbench(meta, val_ids, picker, n_trials=30, seed=20, sample_rate=DEFAULT_SAMPLE_RATE):
    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=seed),
    )
    study.enqueue_trial(
        {
            "P_threshold": 0.075,
            "S_threshold": 0.1,
            "dbscan_eps": 25,
            "dbscan_min_samples": 3,
            "min_picks_per_eq": 5,
        }
    )
    study.optimize(
        lambda t: objective(t, meta, val_ids, picker, sample_rate=sample_rate),
        n_trials=n_trials,
    )

    best = study.best_trial
    best_params = SBParams(**best.params)
    best_metrics = best.user_attrs["metrics"]
    trials_df = pd.DataFrame(
        [
            {
                **t.params,
                **(t.user_attrs.get("metrics", {}) if t.user_attrs else {}),
                "objective": t.value,
            }
            for t in study.trials
        ]
    )
    return best_params, best_metrics, trials_df, study



In [ ]:
meta = pd.read_csv("data/chunk4.csv", sep=",")
meta["trace_start_time"] = pd.to_datetime(meta["trace_start_time"], format="mixed", errors="coerce")
meta = meta.drop_duplicates(
    subset=[
        "source_id",
        "network_code",
        "receiver_code",
        "receiver_latitude",
        "receiver_longitude",
        "receiver_elevation_m",
    ]
)

with open("PLAN/Notebook/source_ids.json", "r") as f:
    d = json.load(f)
    val_ids = d["val"]
    test_ids = d["test"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
picker = sbm.EQTransformer.from_pretrained("instance").to(device)
if device.type == "cuda":
    picker.cuda()

/tmp/ipykernel_13308/3879882903.py:1: DtypeWarning: Columns (0: source_mechanism_strike_dip_rake) have mixed types. Specify dtype option on import or set low_memory=False.
  meta = pd.read_csv("data/chunk4.csv", sep=",")


In [ ]:
total_params = sum(p.numel() for p in picker.parameters())
total_params

376935

# optimization

In [ ]:
import logging

logging.getLogger("seisbench").setLevel(logging.ERROR)
import os
os.environ["TQDM_DISABLE"] = "1"

In [ ]:


best_params, best_val_metrics, trials_df, study = optimize_seisbench(
    meta=meta,
    val_ids=val_ids[:50],
    picker=picker,
    n_trials=30,
    seed=20,
)



[I 2026-05-22 13:28:43,100] A new study created in memory with name: no-name-c7c6a8b6-d330-405b-8ef3-0b6f333502f2
Clustering (eps=20.83): 100%|██████████████████| 1/1 [00:00<00:00, 10131.17it/s]


Associating 11 picks with 1 CPUs
.

Clustering (eps=20.83): 100%|██████████████████| 2/2 [00:00<00:00, 21024.08it/s]


Associating 7 picks with 1 CPUs
.

Clustering (eps=20.83): 100%|██████████████████| 1/1 [00:00<00:00, 10180.35it/s]


...

Associating 7 picks with 1 CPUs
.

Clustering (eps=20.00): 100%|██████████████████| 1/1 [00:00<00:00, 15650.39it/s]


Associating 8 picks with 1 CPUs
.

Clustering (eps=20.00): 100%|██████████████████| 1/1 [00:00<00:00, 13617.87it/s]
[I 2026-05-22 13:32:46,723] Trial 29 finished with value: 3.4288910130671697 and parameters: {'P_threshold': 0.07436746271289872, 'S_threshold': 0.10937616007871796, 'dbscan_eps': 24, 'dbscan_min_samples': 3, 'min_picks_per_eq': 5}. Best is trial 23 with value: 2.3597820599274333.


Associating 14 picks with 1 CPUs
.

In [ ]:
test_metrics, test_evals = evaluate_seisbench(
    meta=meta,
    source_ids=test_ids[:500],
    picker=picker,
    params=best_params,
)

test_evals_clean = test_evals[test_evals.apply(lambda row: all(row != 1e6), axis=1)]

Clustering (eps=27.50): 100%|██████████████████| 1/1 [00:00<00:00, 14217.98it/s]


Associating 8 picks with 1 CPUs
.

Clustering (eps=27.50): 100%|██████████████████| 1/1 [00:00<00:00, 15141.89it/s]


Associating 8 picks with 1 CPUs
.

Clustering (eps=27.50): 100%|██████████████████| 1/1 [00:00<00:00, 14563.56it/s]


...

Associating 8 picks with 1 CPUs
.

Clustering (eps=27.50): 100%|██████████████████| 2/2 [00:00<00:00, 31418.01it/s]


Associating 6 picks with 1 CPUs
.

Clustering (eps=27.50): 100%|██████████████████| 1/1 [00:00<00:00, 11459.85it/s]

Associating 7 picks with 1 CPUs
.

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=2, cols=2, subplot_titles=('count_penalty', 'dt', 'dist', 'depth'))
for i, col in enumerate(['count_penalty', 'dt', 'dist', 'depth']):
    fig.add_trace(
        go.Histogram(x=test_evals_clean[col], name=col),
        row=i // 2 + 1,
        col=i % 2 + 1,
    )

fig.update_layout(
    height=1000,
    width=1000,
    title="Histograms of EQ+GAMMA evaluation metrics",
)

fig.show()


In [ ]:
test_evals_clean.mean()

count_penalty     0.028455
dt                5.769230
dist             44.013876
depth            36.678500
dtype: float64

In [ ]:
test_evals_clean.mean().round(3)

count_penalty     0.028
dt                5.769
dist             44.014
depth            36.678
dtype: float64

In [ ]:
count_penalty     1.937282
dt                4.848746
dist             27.865168
depth             5.596526
dtype: float64